# T2SMark SD3.5 one-unit GPU canary

Experiment-log handoff only. It records one real clean/watermarked physical unit and six attacked conditions; it does not establish robustness, a threshold, TPR/FPR, or a scientific claim.

## Goal / scope
Run the official T2SMark SD3.5 route from ordinary RGB through VAE mode, inversion, key channels, and native `norm1_w`. The only permitted output claim is `engineering_canary_complete` when all twelve real score calls finish.

## Drive mount
The first executable cell is intentionally the independent two-line mount cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import secrets, json, traceback
root = Path('/content/drive/MyDrive/CEG-WM/Baseline-V1/T2SMark-Canary')
run_dir = root / (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + secrets.token_hex(4))
run_dir.mkdir(parents=True, exist_ok=False)
(run_dir / 'run_started.json').write_text('{"status": "started", "planned_observations": 12}', encoding='utf-8')
def record_failure(stage, exc):
    failure = {'status':'failed', 'stage':stage, 'failure':f'{type(exc).__name__}: {exc}', 'planned_observations':12, 'observations':[{'condition':c, 'role':r, 'status':'failed', 'failure':'run stopped before this observation'} for c in ('clean_no_attack','jpeg_q50','resize_50_bicubic_restore','center_crop_80_restore','gaussian_blur_sigma_1px','rotation_10_bicubic_reflect_center_crop_v1') for r in ('clean_negative','watermarked_positive')]}
    (run_dir / f'run_failed_{secrets.token_hex(4)}.json').write_text(json.dumps(failure, indent=2), encoding='utf-8')
def _record_uncaught(shell, exc_type, exc, tb, tb_offset=None):
    record_failure('unhandled_notebook_cell', exc)
    return None
get_ipython().set_custom_exc((BaseException,), _record_uncaught)
print(run_dir)

## Parameters and dependencies
The notebook is self-contained: it fetches the official source at its pinned exact and does not require this local branch to be pushed.

In [ ]:
import os, sys, json, math, secrets, subprocess, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter
import torch

OFFICIAL_REPO = 'https://github.com/0xD009/T2SMark.git'
OFFICIAL_EXACT = '0c1fbfd50fcd1fba135477a2c016e284d5d7914d'
LOCAL_ADAPTER_EXACT = 'a7d814431e11f326ccee2f417bd1a4d7e5720555'
MODEL_ID = 'stabilityai/stable-diffusion-3.5-medium'
MODEL_REVISION = 'b940f670f0eda2d07fbb75229e779da1ad11eb80'
PROMPT = 'A small red ceramic cube on a pale wooden table, studio photograph'
GENERATION_SEED, WATERMARK_SEED = 1701, 9173
KEY_LENGTH, MESSAGE_LENGTH, TAU = 16, 256, 0.674
GUIDANCE_SCALE, NUM_INFERENCE_STEPS, NUM_INVERSION_STEPS = 4.0, 40, 10
LATENT_SHAPE, KEY_CHANNELS, MESSAGE_CHANNELS = (1, 16, 64, 64), (0, 1, 2, 3), tuple(range(4, 16))
assert torch.cuda.is_available(), 'CUDA is required: stop rather than substitute a weaker model or synthetic fallback.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'diffusers==0.32.0', 'transformers==4.45.2', 'accelerate==1.1.1', 'huggingface_hub==0.26.2', 'safetensors==0.4.5', 'sentencepiece==0.2.0'], check=True)
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, 'Set the Colab HF_TOKEN secret with gated-model access.'

## Official source
Clone into a new `/content` directory and detached-checkout the reviewed official exact.

In [ ]:
source_dir = Path('/content') / f't2smark-{secrets.token_hex(4)}'
subprocess.run(['git', 'clone', OFFICIAL_REPO, str(source_dir)], check=True)
subprocess.run(['git', '-C', str(source_dir), 'checkout', '--detach', OFFICIAL_EXACT], check=True)
assert (source_dir / 'src/inversion/inverse_diffusion3.py').is_file()
sys.path.insert(0, str(source_dir))
from src.inversion.inverse_diffusion3 import InversionDiffusion3Pipeline

## Method core
This is the reviewed T2S-A codec core, synchronized from `src/cegwm/baselines/t2smark.py`. Keyed signs and support use the official CPU PRNG topology before transfer to the latent device.

In [ ]:
from functools import reduce
from operator import mul

def _bits(bits, length, device):
    value = torch.as_tensor(bits, device=device, dtype=torch.int64)
    assert value.ndim == 1 and value.numel() == length and bool(torch.all((value == 0) | (value == 1)))
    return value

def _bits_to_int(bits):
    return int(reduce(lambda acc, bit: acc * 2 + int(bit), bits.tolist(), 0))

class T2SMarkCodec:
    def __init__(self, message_length, tau, latent_shape):
        self.message_length, self.tau, self.latent_shape = message_length, tau, tuple(latent_shape)
        self.noise_size = reduce(mul, self.latent_shape, 1)
        self.repeat_count = int(math.erfc(tau / math.sqrt(2.0)) * self.noise_size / message_length)
        self.codeword_length = self.message_length * self.repeat_count
        assert self.repeat_count > 0 and self.codeword_length <= self.noise_size
    def _support_and_signs(self, key_bits, device, dtype):
        key = torch.as_tensor(key_bits, dtype=torch.int64)
        assert key.ndim == 1 and key.numel() > 0 and bool(torch.all((key == 0) | (key == 1)))
        generator = torch.Generator()
        generator.manual_seed(_bits_to_int(key))
        signs = torch.randint(0, 2, (self.codeword_length,), generator=generator, device='cpu')
        support = torch.randperm(self.noise_size, generator=generator, device='cpu')[:self.codeword_length]
        return support.to(device), signs.to(device=device, dtype=dtype).mul(2).sub(1)
    def encode(self, bits, key_bits, base_noise):
        message = _bits(bits, self.message_length, base_noise.device)
        support, keyed_signs = self._support_and_signs(key_bits, base_noise.device, base_noise.dtype)
        selector = torch.zeros(self.noise_size, dtype=torch.bool, device=base_noise.device); selector[support] = True
        codeword = (1 - 2 * message).repeat(self.repeat_count).to(base_noise.dtype) * keyed_signs
        flat = base_noise.flatten(); tail = torch.topk(flat.abs(), self.codeword_length, largest=True, sorted=False); central = torch.topk(flat.abs(), self.noise_size-self.codeword_length, largest=False, sorted=False)
        out = torch.empty_like(flat); out[selector] = tail.values * codeword
        signs = torch.randint(0, 2, (self.noise_size-self.codeword_length,), device=base_noise.device).to(base_noise.dtype).mul(2).sub(1)
        out[~selector] = central.values * signs
        return out.reshape(self.latent_shape)
    def decode(self, reversed_noise, key_bits):
        support, signs = self._support_and_signs(key_bits, reversed_noise.device, reversed_noise.dtype)
        selector = torch.zeros(self.noise_size, dtype=torch.bool, device=reversed_noise.device); selector[support] = True
        p = (reversed_noise.flatten()[selector] * signs).reshape(self.repeat_count, self.message_length).sum(0)
        return (p < 0).to(torch.int64), float(torch.linalg.vector_norm(p, ord=1).item())

def embed_t2smark_sd35(base_latent, master_key_bits, session_key_bits, message_bits):
    master, session, message = _bits(master_key_bits, 16, base_latent.device), _bits(session_key_bits, 16, base_latent.device), _bits(message_bits, 256, base_latent.device)
    key_codec, msg_codec = T2SMarkCodec(16, TAU, (4,64,64)), T2SMarkCodec(256, TAU, (12,64,64))
    result = base_latent.clone(); result[0, KEY_CHANNELS] = key_codec.encode(session, master, base_latent[0, KEY_CHANNELS]); result[0, MESSAGE_CHANNELS] = msg_codec.encode(message, session, base_latent[0, MESSAGE_CHANNELS])
    return result

def score_t2smark_rgb(rgb, pipeline, master_key_bits):
    assert isinstance(rgb, np.ndarray) and rgb.dtype == np.uint8 and rgb.ndim == 3 and rgb.shape[2] == 3
    device = torch.device(pipeline._execution_device)
    image = torch.from_numpy(rgb.copy()).permute(2,0,1).unsqueeze(0).to(device=device, dtype=torch.float16).div(255).mul(2).sub(1)
    latents = pipeline.get_image_latents(image, sample=False)
    reversed_latents = pipeline.naive_forward_diffusion(latents=latents, num_inference_steps=NUM_INVERSION_STEPS)
    _, norm1_w = T2SMarkCodec(16, TAU, (4,64,64)).decode(reversed_latents[0, KEY_CHANNELS], _bits(master_key_bits, 16, device))
    assert math.isfinite(norm1_w)
    return norm1_w

## Model load and one-unit generation
Both images receive the same prompt, scheduler settings, and base latent; only the watermark transform differs.

In [ ]:
pipe = InversionDiffusion3Pipeline.from_pretrained(MODEL_ID, revision=MODEL_REVISION, torch_dtype=torch.float16, token=HF_TOKEN)
try:
    pipe = pipe.to('cuda')
except torch.cuda.OutOfMemoryError:
    pipe.enable_model_cpu_offload()
pipe.set_progress_bar_config(disable=True)
device = torch.device(pipe._execution_device)
latent_generator = torch.Generator(device='cuda').manual_seed(GENERATION_SEED)
base_latent = torch.randn(LATENT_SHAPE, generator=latent_generator, device=device, dtype=torch.float16)
key_generator = torch.Generator(device='cuda').manual_seed(WATERMARK_SEED)
master_key = torch.randint(0, 2, (16,), generator=key_generator, device=device)
session_key = torch.randint(0, 2, (16,), generator=key_generator, device=device)
message_bits = torch.randint(0, 2, (256,), generator=key_generator, device=device)
watermarked_latent = embed_t2smark_sd35(base_latent, master_key, session_key, message_bits)
common = dict(prompt=PROMPT, guidance_scale=GUIDANCE_SCALE, num_inference_steps=NUM_INFERENCE_STEPS, height=512, width=512)
clean_image = pipe(latents=base_latent, **common).images[0].convert('RGB')
watermarked_image = pipe(latents=watermarked_latent, **common).images[0].convert('RGB')
del base_latent, watermarked_latent
torch.cuda.empty_cache()

## Attacks and real scoring
Each condition function is applied identically to the clean negative and watermarked positive image. Rotation is +10 degree visual CCW about pixel center with reflected padding, bicubic RGB, nearest zero mask, then center crop.

In [ ]:
def _rgb(image): return np.asarray(image.convert('RGB'), dtype=np.uint8)
def jpeg_q50(image):
    import io
    b = io.BytesIO(); image.save(b, format='JPEG', quality=50, subsampling=2, optimize=False, progressive=False); return Image.open(io.BytesIO(b.getvalue())).convert('RGB')
def resize_50_bicubic_restore(image):
    w,h=image.size; small_w,small_h=max(1, round(w * 0.50)),max(1, round(h * 0.50)); return image.resize((small_w,small_h), Image.Resampling.BICUBIC).resize((w,h), Image.Resampling.BICUBIC)
def center_crop_80_restore(image):
    w,h=image.size; scale=math.sqrt(0.80); crop_w,crop_h=max(1,round(w*scale)),max(1,round(h*scale)); left,top=(w-crop_w)//2,(h-crop_h)//2; return image.crop((left,top,left+crop_w,top+crop_h)).resize((w,h), Image.Resampling.BICUBIC)
def gaussian_blur_sigma_1px(image): return image.filter(ImageFilter.GaussianBlur(radius=1.0))
def rotation_10_bicubic_reflect_center_crop_v1(image):
    source=_rgb(image); h,w=source.shape[:2]; theta=math.radians(10.0); a,b=(w-1)/2.0,(h-1)/2.0; p_x=max(0,math.ceil(abs(math.cos(theta))*a+abs(math.sin(theta))*b+2-a)); p_y=max(0,math.ceil(abs(math.sin(theta))*a+abs(math.cos(theta))*b+2-b)); assert p_x<w and p_y<h
    padded=np.pad(source,((p_y,p_y),(p_x,p_x),(0,0)),mode='reflect'); center=(p_x+(w-1)/2.0,p_y+(h-1)/2.0); box=(p_x,p_y,p_x+w,p_y+h)
    rgb=Image.fromarray(padded).rotate(10.0,resample=Image.Resampling.BICUBIC,expand=False,center=center,fillcolor=(0,0,0)); mask=np.pad(np.ones((h,w),np.uint8),((p_y,p_y),(p_x,p_x)),mode='constant',constant_values=0); valid=Image.fromarray(mask,mode='L').rotate(10.0,resample=Image.Resampling.NEAREST,expand=False,center=center,fillcolor=0)
    return rgb.crop(box), np.asarray(valid.crop(box),dtype=np.uint8)
def apply_rotation_rgb(image): return rotation_10_bicubic_reflect_center_crop_v1(image)[0]
conditions = {'clean_no_attack': lambda x:x, 'jpeg_q50':jpeg_q50, 'resize_50_bicubic_restore':resize_50_bicubic_restore, 'center_crop_80_restore':center_crop_80_restore, 'gaussian_blur_sigma_1px':gaussian_blur_sigma_1px, 'rotation_10_bicubic_reflect_center_crop_v1':apply_rotation_rgb}
planned_observations, rows, artifacts = 12, [], {}
for condition_name, condition in conditions.items():
    for role, image in [('clean_negative', clean_image), ('watermarked_positive', watermarked_image)]:
        started=time.time(); attacked=condition(image); artifacts[(condition_name, role)] = attacked
        try:
            score=score_t2smark_rgb(_rgb(attacked), pipe, master_key); status, failure='ok', None
        except Exception as exc:
            score, status, failure=None, 'failed', f'{type(exc).__name__}: {exc}'
        rows.append({'condition':condition_name, 'role':role, 'score':score, 'runtime_seconds':time.time()-started, 'status':status, 'failure':failure})
assert len(rows) == planned_observations

## Create-only output
A new timestamp-and-random-suffix directory is created every run. No existing Drive content is removed or overwritten.

In [ ]:
clean_image.save(run_dir / 'clean.png'); watermarked_image.save(run_dir / 'watermarked.png')
for (condition_name, role), image in artifacts.items():
    if condition_name != 'clean_no_attack': image.save(run_dir / f'{condition_name}_{role}.png')
scores = pd.DataFrame(rows); scores.to_csv(run_dir / 'scores.csv', index=False)
result = {'method':'T2SMark SD3.5', 'model_id':MODEL_ID, 'model_revision':MODEL_REVISION, 'official_source_exact':OFFICIAL_EXACT, 'local_adapter_exact':LOCAL_ADAPTER_EXACT, 'prompt':PROMPT, 'generation_seed':GENERATION_SEED, 'watermark_seed':WATERMARK_SEED, 'device_name':torch.cuda.get_device_name(), 'parameters':{'height':512,'width':512,'guidance_scale':GUIDANCE_SCALE,'num_inference_steps':NUM_INFERENCE_STEPS,'num_inversion_steps':NUM_INVERSION_STEPS,'key_length':16,'message_length':256,'tau':TAU}, 'planned_observations':planned_observations, 'observations':rows, 'engineering_canary_complete':all(r['status']=='ok' for r in rows)}
(run_dir / 'canary_result.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
print(run_dir); display(scores[['condition','role','score','runtime_seconds','status']])

## Checks and next steps
A completed engineering canary means only that all 12 real calls finished. Do not infer robustness, superiority, a threshold, TPR/FPR, or a paper result. Inspect the immutable Drive artifact before selecting any later evaluation stage.